# YOLOv8n Ablation B / C / D — Source-Grouped 5-Fold CV (Google Colab)

Compares three training changes against the **source-grouped CV baseline (A)** using the **exact same folds and seeds**,
so every comparison is paired fold-by-fold.

| Run | Model | Change vs. baseline A | Why |
|---|---|---|---|
| A | `yolov8n.pt` | — (loaded from `leak_free_yolov8n_srcgroup_cv_v1`) | baseline: mAP50 0.695 ± 0.039 |
| B | `yolov8n.pt` | `flipud=0.5`, `epochs=100`, `patience=50` | top-down imagery has no "up"; Folds 2/5 stopped too early |
| C | `yolov8n.pt` | B + `hsv_s=0.3`, `hsv_v=0.2` | weaker colour jitter may preserve disease colour cues |
| D | `yolov8s.pt` | augmentation of the better of B/C | more capacity, full COCO weight transfer |

Not included (by design): P2 head (all boxes are 32–73 px, mean ≈ 54 px — no small objects) and
SE/ECA attention (equivalent to the LiteSE already tried).

### Protocol
- Folds, leak groups (source filename ∪ exact pHash) and seeds (43–47) are **read from the baseline experiment**.
- Each fold model is evaluated **only on its own held-out fold**, at a fixed **640 × 640** input (`rect=False`).
- The 42-image independent test set is **not used** in this notebook. Pick the winner by CV, then test it once.
- Output: per-fold metrics, paired differences vs. A, number of folds improved, paired t-test.

### Outputs (Google Drive)
`MyDrive/ganoderma/experiments/ablation_yolov8n_srcgroup_v1/runs/<run>/fold_<k>/` and `summary/`.

**Runtime (A100):** B and C ≈ 4–8 min per fold, D ≈ 8–15 min per fold → roughly 1.5–2.5 h total.
Completed run/fold pairs are skipped on rerun, so you can stop and resume any time.

In [ ]:
# ============================================================
# 0. INSTALL DEPENDENCIES
# ============================================================
!pip -q install "ultralytics==8.4.173" "scikit-learn>=1.4" scipy pyyaml tqdm

import ultralytics, torch
print("Ultralytics:", ultralytics.__version__, "| PyTorch:", torch.__version__)
assert ultralytics.__version__ == "8.4.173", "Pin ultralytics to 8.4.173 (same as the baseline CV)"
if not torch.cuda.is_available():
    raise RuntimeError("GPU is not available. Runtime > Change runtime type > GPU, then reconnect VS Code.")
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# ============================================================
# 1. MOUNT DRIVE + CONFIGURATION
# ============================================================
from google.colab import drive
drive.mount("/content/drive")
from pathlib import Path

DATASET_ROOT = Path("/content/drive/MyDrive/ganoderma")
VAL_FOLDER_NAME = "valid" if (DATASET_ROOT / "valid").exists() else "val"

# Baseline experiment: provides folds, leak groups, seeds and baseline-A metrics
BASE_EXP  = "leak_free_yolov8n_srcgroup_cv_v1"
BASE_ROOT = DATASET_ROOT / "experiments" / BASE_EXP

EXP_NAME = "ablation_yolov8n_srcgroup_v1"
EXP_ROOT = DATASET_ROOT / "experiments" / EXP_NAME
EXP_ROOT.mkdir(parents=True, exist_ok=True)

LOCAL_DATASET = Path("/content/ganoderma_source")
LOCAL_WORK    = Path("/content/ganoderma_ablation_work")

SEED    = 42           # fold seed = SEED + fold (identical to baseline)
N_FOLDS = 5
IMGSZ   = 640
DEVICE  = 0
WORKERS = 2

# Common training settings (identical to baseline unless overridden per run)
COMMON = dict(imgsz=IMGSZ, batch=32, workers=WORKERS, device=DEVICE, deterministic=True, pretrained=True)

# Ablation runs (executed in this order; D must come after B and C)
RUNS = {
    "B_aerial":          dict(model="yolov8n.pt", epochs=100, patience=50, flipud=0.5),
    "C_aerial_lowcolor": dict(model="yolov8n.pt", epochs=100, patience=50, flipud=0.5, hsv_s=0.3, hsv_v=0.2),
    "D_yolov8s":         dict(model="yolov8s.pt", epochs=100, patience=50, aug_from="auto"),
}
RUNS_ENABLED = ["B_aerial", "C_aerial_lowcolor", "D_yolov8s"]

# D augmentation: "auto" = better of B/C by mean held-out mAP50 (needs all 5 folds of B and C),
#                 or force "B_aerial" / "C_aerial_lowcolor"
D_AUG = "auto"
AUG_KEYS = ["flipud", "fliplr", "hsv_h", "hsv_s", "hsv_v", "degrees", "translate", "scale", "mosaic", "mixup"]

# Held-out evaluation protocol (same as baseline)
EVAL = dict(imgsz=IMGSZ, rect=False, conf=0.001, iou=0.7, max_det=300)

FOLDS_TO_RUN = list(range(1, N_FOLDS + 1))
RESUME_COMPLETED = True
KEEP_LOCAL_RUN_DIRS = False

SMOKE_TEST = False
if SMOKE_TEST:
    EXP_NAME += "_smoke"; EXP_ROOT = DATASET_ROOT / "experiments" / EXP_NAME; EXP_ROOT.mkdir(parents=True, exist_ok=True)
    FOLDS_TO_RUN = [1]
    for r in RUNS.values():
        r["epochs"] = 1
    D_AUG = "B_aerial"

print("Baseline :", BASE_ROOT)
print("Ablation :", EXP_ROOT)
print("Runs     :", RUNS_ENABLED)
print("Folds    :", FOLDS_TO_RUN)

In [ ]:
# ============================================================
# 2. IMPORTS + HELPERS
# ============================================================
import os, gc, json, time, shutil, random, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from ultralytics import YOLO
from ultralytics.utils.torch_utils import get_flops

warnings.filterwarnings("ignore", category=UserWarning)
SUPPORTED_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}
METRICS = ["mAP50", "mAP50-95", "Precision", "Recall", "F1"]

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def extract_metrics(res):
    g = lambda n: float(getattr(res.box, n)() if callable(getattr(res.box, n)) else getattr(res.box, n))
    p, r = g("mp"), g("mr")
    return {"mAP50": g("map50"), "mAP50-95": g("map"), "Precision": p, "Recall": r,
            "F1": 2 * p * r / (p + r + 1e-12)}

def write_json(obj, path):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    json.dump(obj, open(path, "w"), indent=2, default=str)

def clear_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def sync_run_lean(local_run_dir, drive_run_dir):
    local_run_dir, drive_run_dir = Path(local_run_dir), Path(drive_run_dir)
    drive_run_dir.mkdir(parents=True, exist_ok=True)
    for item in local_run_dir.iterdir():
        if item.name == "weights":
            continue
        dst = drive_run_dir / item.name
        if dst.exists():
            shutil.rmtree(dst) if dst.is_dir() else dst.unlink()
        shutil.copytree(item, dst) if item.is_dir() else shutil.copy2(item, dst)

write_json({"base_exp": BASE_EXP, "runs": RUNS, "runs_enabled": RUNS_ENABLED, "common": COMMON,
            "eval": EVAL, "d_aug": D_AUG, "seed": SEED, "folds": FOLDS_TO_RUN,
            "ultralytics": ultralytics.__version__, "torch": torch.__version__,
            "gpu": torch.cuda.get_device_name(0)}, EXP_ROOT / "experiment_config.json")
print("config saved")

In [ ]:
# ============================================================
# 3. LOCAL DATA CACHE + BASELINE FOLDS / METRICS
# ============================================================
LOCAL_DATASET.mkdir(parents=True, exist_ok=True)
for src, dst in [("train", "train"), (VAL_FOLDER_NAME, "valid")]:   # test split NOT copied (not used here)
    if (LOCAL_DATASET / dst).exists():
        shutil.rmtree(LOCAL_DATASET / dst)
    shutil.copytree(DATASET_ROOT / src, LOCAL_DATASET / dst)
print("local cache ready:", LOCAL_DATASET)

SPLITS_DIR = BASE_ROOT / "splits"
folds = {}
for k in range(1, N_FOLDS + 1):
    tr, va, te = (pd.read_csv(SPLITS_DIR / f"fold_{k}_{p}.csv") for p in ["train", "val", "test"])
    for a, b in [(tr, va), (tr, te), (va, te)]:
        assert not (set(a["uid"]) & set(b["uid"]))
        assert not (set(a["leak_group"]) & set(b["leak_group"]))
    assert not any(s == "test" for s in pd.concat([tr, va, te])["source_split"])
    folds[k] = (tr, va, te)
print(pd.read_csv(SPLITS_DIR / "fold_split_summary.csv").to_string(index=False))

base_rows = []
for k in range(1, N_FOLDS + 1):
    m = pd.read_csv(BASE_ROOT / "models" / "yolov8n" / "folds" / f"fold_{k}" / "metrics.csv").iloc[0]
    base_rows.append({"run": "A_baseline", "fold": k, **{c: float(m[c]) for c in METRICS},
                      "params": int(m["params"]), "gflops": float(m["gflops"]), "epochs_run": np.nan})
base_df = pd.DataFrame(base_rows)
print("\nBaseline A (held-out, 640x640):")
print(base_df[["fold"] + METRICS].round(4).to_string(index=False))
print("mean mAP50 = %.4f +/- %.4f" % (base_df["mAP50"].mean(), base_df["mAP50"].std(ddof=1)))

In [ ]:
# ============================================================
# 4. TRAIN + EVALUATE B / C / D
# ============================================================
def materialize(df, out_root):
    img_dir, lbl_dir = Path(out_root) / "images", Path(out_root) / "labels"
    img_dir.mkdir(parents=True, exist_ok=True); lbl_dir.mkdir(parents=True, exist_ok=True)
    for _, r in df.iterrows():
        name = f"{r['source_split']}__{r['filename']}"
        shutil.copy2(LOCAL_DATASET / r["rel_image"], img_dir / name)
        src = LOCAL_DATASET / r["rel_label"]; dst = lbl_dir / f"{Path(name).stem}.txt"
        shutil.copy2(src, dst) if src.exists() else dst.write_text("")

def fold_dataset(k):
    # Materialize each fold once and reuse it across runs.
    d = LOCAL_WORK / "data" / f"fold_{k}"
    y = d / "data.yaml"
    if y.exists():
        return y
    tr, va, te = folds[k]
    for name, df in [("train", tr), ("val", va), ("test", te)]:
        materialize(df, d / name)
        n = len([p for p in (d / name / "images").iterdir() if p.suffix.lower() in SUPPORTED_EXTS])
        assert n == len(df), (k, name, n, len(df))
    y.write_text(f"train: {d/'train'/'images'}\nval: {d/'val'/'images'}\ntest: {d/'test'/'images'}\n"
                 f"nc: 1\nnames: ['Ganoderma']\n")
    return y

def run_done(run, k):
    d = EXP_ROOT / "runs" / run / f"fold_{k}"
    return (d / "best.pt").exists() and (d / "metrics.csv").exists()

def run_mean_map50(run):
    vals = []
    for k in range(1, N_FOLDS + 1):
        p = EXP_ROOT / "runs" / run / f"fold_{k}" / "metrics.csv"
        if not p.exists():
            return None
        vals.append(float(pd.read_csv(p)["mAP50"].iloc[0]))
    return float(np.mean(vals))

def resolve_train_args(run):
    cfg = dict(RUNS[run])
    aug_from = cfg.pop("aug_from", None)
    if aug_from is not None:
        src = D_AUG
        if src == "auto":
            mb, mc = run_mean_map50("B_aerial"), run_mean_map50("C_aerial_lowcolor")
            if mb is None or mc is None:
                raise RuntimeError("D_AUG='auto' needs all 5 folds of B and C. Finish them or force D_AUG.")
            src = "B_aerial" if mb >= mc else "C_aerial_lowcolor"
            print(f"  D augmentation: {src}  (B mAP50={mb:.4f}, C mAP50={mc:.4f})")
        for key in AUG_KEYS:
            if key in RUNS[src]:
                cfg[key] = RUNS[src][key]
        write_json({"d_aug_source": src, "args": cfg}, EXP_ROOT / "runs" / run / "d_aug_choice.json")
    model = cfg.pop("model")
    return model, cfg

all_rows = []
for run in RUNS_ENABLED:
    print("\n" + "#" * 90 + f"\nRUN: {run}\n" + "#" * 90)
    model_cfg, train_cfg = resolve_train_args(run)
    print("  model:", model_cfg, "| args:", train_cfg)
    for k in FOLDS_TO_RUN:
        drive_dir = EXP_ROOT / "runs" / run / f"fold_{k}"
        if RESUME_COMPLETED and run_done(run, k):
            row = pd.read_csv(drive_dir / "metrics.csv").iloc[0].to_dict()
            all_rows.append(row)
            print(f"  fold {k}: already done (mAP50={row['mAP50']:.4f}) -> skip")
            continue

        yaml_path = fold_dataset(k)
        local_dir = LOCAL_WORK / run / f"fold_{k}"
        shutil.rmtree(local_dir, ignore_errors=True); local_dir.mkdir(parents=True, exist_ok=True)

        seed_everything(SEED + k)
        t0 = time.time()
        model = YOLO(model_cfg)
        model.train(data=str(yaml_path), seed=SEED + k, project=str(local_dir), name="train_run",
                    exist_ok=True, verbose=False, plots=True, **COMMON, **train_cfg)
        minutes = (time.time() - t0) / 60

        best_pt = local_dir / "train_run" / "weights" / "best.pt"
        res_csv = pd.read_csv(local_dir / "train_run" / "results.csv")
        best = YOLO(str(best_pt))
        ev = best.val(data=str(yaml_path), split="test", batch=32, device=DEVICE,
                      project=str(local_dir), name="heldout_eval", exist_ok=True, plots=True, verbose=False, **EVAL)
        m = extract_metrics(ev)
        try:
            gfl = float(get_flops(best.model, IMGSZ))
        except Exception:
            gfl = float("nan")
        row = {"run": run, "fold": k, **m,
               "params": int(sum(p.numel() for p in best.model.parameters())), "gflops": round(gfl, 1),
               "epochs_run": int(len(res_csv)), "train_minutes": round(minutes, 2), "seed": SEED + k,
               "model": model_cfg, "train_args": json.dumps(train_cfg)}
        drive_dir.mkdir(parents=True, exist_ok=True)
        pd.DataFrame([row]).to_csv(drive_dir / "metrics.csv", index=False)
        shutil.copy2(best_pt, drive_dir / "best.pt")
        sync_run_lean(local_dir / "train_run", drive_dir / "train_run")
        sync_run_lean(local_dir / "heldout_eval", drive_dir / "heldout_eval")
        all_rows.append(row)
        print(f"  fold {k}: mAP50={m['mAP50']:.4f}  mAP50-95={m['mAP50-95']:.4f}  "
              f"P={m['Precision']:.4f}  R={m['Recall']:.4f}  epochs={row['epochs_run']}  ({minutes:.1f} min)")
        del model, best, ev
        clear_cuda()
        if not KEEP_LOCAL_RUN_DIRS:
            shutil.rmtree(local_dir, ignore_errors=True)

print("\nAll requested runs complete")

In [ ]:
# ============================================================
# 5. SUMMARY + PAIRED COMPARISON VS BASELINE A
# ============================================================
SUMMARY = EXP_ROOT / "summary"; SUMMARY.mkdir(parents=True, exist_ok=True)

rows = []
for run in RUNS_ENABLED:
    for k in range(1, N_FOLDS + 1):
        p = EXP_ROOT / "runs" / run / f"fold_{k}" / "metrics.csv"
        if p.exists():
            rows.append(pd.read_csv(p).iloc[0].to_dict())
res = pd.concat([base_df, pd.DataFrame(rows)], ignore_index=True)
res.to_csv(SUMMARY / "per_fold_metrics.csv", index=False)

print("PER-FOLD HELD-OUT RESULTS (640x640)")
print(res.pivot(index="fold", columns="run", values="mAP50").round(4).to_string())

summ = []
for run, g in res.groupby("run", sort=False):
    r = {"run": run, "n_folds": len(g), "params_M": round(g["params"].iloc[0] / 1e6, 2),
         "GFLOPs": g["gflops"].iloc[0]}
    for c in METRICS:
        r[c] = f"{g[c].mean():.4f} ± {g[c].std(ddof=1):.4f}" if len(g) > 1 else f"{g[c].mean():.4f}"
    if "epochs_run" in g and g["epochs_run"].notna().any():
        r["epochs_run"] = "/".join(str(int(e)) for e in g.sort_values("fold")["epochs_run"])
    summ.append(r)
summ_df = pd.DataFrame(summ)
summ_df.to_csv(SUMMARY / "summary_mean_sd.csv", index=False)
print("\nMEAN ± SD")
print(summ_df.to_string(index=False))

paired = []
for run in RUNS_ENABLED:
    g = res[res["run"] == run].set_index("fold")
    common = sorted(set(g.index) & set(base_df["fold"]))
    if len(common) < 2:
        continue
    a = base_df.set_index("fold").loc[common]
    for c in ["mAP50", "mAP50-95", "Recall", "F1"]:
        d = g.loc[common, c].values - a[c].values
        t, p = stats.ttest_rel(g.loc[common, c].values, a[c].values)
        paired.append({"run": run, "metric": c, "folds": len(common),
                       "mean_diff": round(float(d.mean()), 4), "sd_diff": round(float(d.std(ddof=1)), 4),
                       "folds_improved": f"{int((d > 0).sum())}/{len(common)}",
                       "paired_t_p": round(float(p), 4),
                       "per_fold_diff": " ".join(f"{x:+.3f}" for x in d)})
paired_df = pd.DataFrame(paired)
paired_df.to_csv(SUMMARY / "paired_vs_baseline.csv", index=False)
print("\nPAIRED DIFFERENCE VS BASELINE A (run − A)")
print(paired_df.to_string(index=False))

fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
for i, c in enumerate(["mAP50", "mAP50-95"]):
    for run, g in res.groupby("run", sort=False):
        g = g.sort_values("fold")
        ax[i].plot(g["fold"], g[c], marker="o", lw=2 if run == "A_baseline" else 1.4,
                   ls="--" if run == "A_baseline" else "-", label=run)
    ax[i].set_xticks(range(1, N_FOLDS + 1)); ax[i].set_xlabel("Held-out fold"); ax[i].set_ylabel(c)
    ax[i].grid(alpha=.25); ax[i].set_title(f"{c} per fold (paired)")
ax[0].legend(fontsize=8)
fig.tight_layout(); fig.savefig(SUMMARY / "paired_per_fold.png", dpi=200); plt.show()

print("\nHow to read: a change is convincing only if it improves most folds (e.g. 5/5 or 4/5) AND the mean")
print("difference is clearly larger than its SD. With 5 folds, p < 0.05 needs a very consistent gain.")
print("Saved to:", SUMMARY)

## Next steps

1. Send `summary/summary_mean_sd.csv` and `summary/paired_vs_baseline.csv` (or the printed output).
2. **Choose by CV only.** If a run wins consistently (most folds improved, mean gain > SD of the differences),
   train a new final model with that recipe (final-model notebook, new `EXP_NAME`, source grouping) and evaluate it
   **once** on the 42-image test set. Report in the paper that the test set was used for two models (baseline and winner).
3. If nothing wins consistently, keep the current final model and report the ablation as a negative result —
   that is still a useful finding for a small dataset.
4. For D (YOLOv8s), also measure QNN HTP latency before adopting it: the accuracy gain must justify the extra cost.